# 04 — Chronological Splits, Baselines, and ML Targets

This notebook prepares the canonical leakage-safe artifacts used by downstream TraceAdvisor modeling.

## Responsibilities

1. Load the point-in-time execution feature artifact produced by Notebook 03.
2. Validate the canonical execution-level schema.
3. Audit source missingness without globally dropping usable rows.
4. Create one deterministic chronological split targeting 70/15/15.
5. Preserve complete `execution_start_time_us` groups at split boundaries.
6. Construct next-occurrence CPU/memory targets without naïve row-wise `shift(-1)` across simultaneous executions.
7. Generate Request, Mean, P90, P95, and P99 historical baselines.
8. Evaluate `min_runs` candidates on validation only with the shared evaluator.
9. Lock the validation-selected cold-start threshold and baseline policy.
10. Save canonical train, validation, test, comparison, and metadata artifacts.

Historical profiles are created in Notebook 03 and are not recomputed here.

The final test partition may be materialized and receive deterministic target/recommendation columns, but **it is not scored or used for any model/policy decision in this notebook**.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd


In [2]:
# Locate repository root.
REPO_ROOT = Path.cwd()

while (
    REPO_ROOT.name != "masters-project"
    and REPO_ROOT.parent != REPO_ROOT
):
    REPO_ROOT = REPO_ROOT.parent

if REPO_ROOT.name != "masters-project":
    raise RuntimeError(
        "Could not locate masters-project repository root."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repository root:", REPO_ROOT)


Repository root: /Users/sonalilonkar/Desktop/295AB/295A/TraceAdvisor/masters-project


In [3]:
from src.features.split_utils import chronological_split

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "processed"
    / "google_subset_v1"
)

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROFILE_FEATURES_PATH = DATA_DIR / "job_metrics_v0.parquet"
TRAIN_FEATURES_PATH = DATA_DIR / "train_features.parquet"
VAL_FEATURES_PATH = DATA_DIR / "val_features.parquet"
TEST_FEATURES_PATH = DATA_DIR / "test_features.parquet"
VALIDATION_COMPARISON_PATH = DATA_DIR / "baseline_validation_comparison.csv"
METADATA_PATH = DATA_DIR / "metadata.json"

print("Data directory:", DATA_DIR)
print("Notebook 03 input:", PROFILE_FEATURES_PATH)


Data directory: /Users/sonalilonkar/Desktop/295AB/295A/TraceAdvisor/masters-project/data/processed/google_subset_v1
Notebook 03 input: /Users/sonalilonkar/Desktop/295AB/295A/TraceAdvisor/masters-project/data/processed/google_subset_v1/job_metrics_v0.parquet


## Canonical N03 → N04 Contract

Notebook 04 consumes `job_metrics_v0.parquet` produced by Notebook 03.

The input must provide:
- execution identity and chronology;
- current request fields;
- current observed peak CPU/memory;
- past-only history depth;
- past-only CPU, memory, and runtime profile statistics.

Notebook 04 does not rebuild historical profiles.


In [4]:
BASE_REQUIRED_COLUMNS = [
    # Identity / chronology
    "execution_id",
    "recurrence_key",
    "execution_start_time_us",
    "execution_end_time_us",

    # Current execution request
    "requested_cpu_ncu",
    "requested_memory_normalized",

    # Current execution observed outcomes
    "peak_cpu_ncu",
    "peak_memory_normalized",

    # Overall historical depth
    "prior_execution_count",

    # CPU historical profile
    "hist_peak_cpu_count",
    "hist_peak_cpu_mean",
    "hist_peak_cpu_std",
    "hist_peak_cpu_p90",
    "hist_peak_cpu_p95",
    "hist_peak_cpu_p99",

    # Memory historical profile
    "hist_peak_memory_count",
    "hist_peak_memory_mean",
    "hist_peak_memory_std",
    "hist_peak_memory_p90",
    "hist_peak_memory_p95",
    "hist_peak_memory_p99",

    # Runtime historical profile
    "hist_runtime_count",
    "hist_runtime_mean",
    "hist_runtime_std",
    "hist_runtime_p90",
    "hist_runtime_p95",
    "hist_runtime_p99",
]


def validate_base_schema(
    df: pd.DataFrame,
) -> None:
    missing_columns = [
        column
        for column in BASE_REQUIRED_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            "Missing required canonical columns: "
            f"{missing_columns}"
        )

    if df["execution_id"].isna().any():
        raise ValueError(
            "execution_id contains missing values."
        )

    if df["execution_id"].duplicated().any():
        raise ValueError(
            "execution_id must be unique."
        )

    if df["recurrence_key"].isna().any():
        raise ValueError(
            "recurrence_key contains missing values."
        )

    if df["execution_start_time_us"].isna().any():
        raise ValueError(
            "execution_start_time_us contains missing values."
        )

    if df["execution_end_time_us"].isna().any():
        raise ValueError(
            "execution_end_time_us contains missing values."
        )

    if (
        df["execution_end_time_us"]
        < df["execution_start_time_us"]
    ).any():
        raise ValueError(
            "Found execution_end_time_us earlier than "
            "execution_start_time_us."
        )

    print("Canonical base schema validation passed.")


## Load Notebook 03 Output


In [5]:
if not PROFILE_FEATURES_PATH.exists():
    raise FileNotFoundError(
        "Notebook 03 output was not found. "
        f"Expected: {PROFILE_FEATURES_PATH}. "
        "Run 03_buildProfilesAndMetrics.ipynb first."
    )

execution_features = pd.read_parquet(
    PROFILE_FEATURES_PATH
)

print(
    "Loaded execution features:",
    execution_features.shape,
)

print(
    "Unique execution IDs:",
    execution_features["execution_id"].nunique(),
)

display(
    execution_features.head()
)


Loaded execution features: (253585, 49)
Unique execution IDs: 253585


,execution_id,collection_id,instance_index,candidate_instance_key,episode_number,execution_start_time_us,execution_end_time_us,observation_count,machine_count,requested_cpu_ncu,...,hist_peak_memory_p90,hist_peak_memory_p95,hist_peak_memory_p99,hist_runtime_count,hist_runtime_mean,hist_runtime_std,hist_runtime_p50,hist_runtime_p90,hist_runtime_p95,hist_runtime_p99
0,83212368472||1008||episode_1,83212368472,1008,83212368472||1008,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
1,83212368472||1042||episode_1,83212368472,1042,83212368472||1042,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
2,83212368472||1056||episode_1,83212368472,1056,83212368472||1056,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
3,83212368472||1096||episode_1,83212368472,1096,83212368472||1096,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
4,83212368472||1099||episode_1,83212368472,1099,83212368472||1099,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
validate_base_schema(
    execution_features
)


Canonical base schema validation passed.


## Missing-Value Audit

Do not apply one global `dropna()` here.

CPU and memory can have different valid-row sets. In particular, missing
memory observations should not automatically remove otherwise valid CPU rows.


In [7]:
important_columns = [
    "requested_cpu_ncu",
    "requested_memory_normalized",
    "peak_cpu_ncu",
    "peak_memory_normalized",
    "prior_execution_count",
    "hist_peak_cpu_count",
    "hist_peak_memory_count",
    "hist_peak_cpu_mean",
    "hist_peak_memory_mean",
    "hist_peak_cpu_p90",
    "hist_peak_memory_p90",
    "hist_peak_cpu_p95",
    "hist_peak_memory_p95",
    "hist_peak_cpu_p99",
    "hist_peak_memory_p99",
]

missing_summary = (
    execution_features[important_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_rows")
    .to_frame()
)

display(missing_summary)


,missing_rows
hist_peak_memory_p99,83158
hist_peak_memory_p90,83158
hist_peak_memory_mean,83158
hist_peak_memory_p95,83158
hist_peak_cpu_p95,80801
hist_peak_cpu_mean,80801
hist_peak_cpu_p90,80801
hist_peak_cpu_p99,80801
peak_memory_normalized,4495
requested_cpu_ncu,774


## Chronological 70/15/15 Candidate Split

Create a deterministic chronological split targeting 70/15/15 while
preserving complete `execution_start_time_us` groups.

Because timestamp groups are never divided across partitions, the actual
row percentages may differ slightly from exactly 70/15/15.

In [8]:
(
    train_df,
    val_df,
    test_df,
    split_metadata,
) = chronological_split(
    execution_features,
    time_col="execution_start_time_us",
    id_col="execution_id",
    train_fraction=0.70,
    val_fraction=0.15,
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


Train: 178858
Validation: 36703
Test: 38024


## Split Integrity Checks


In [9]:
assert len(train_df) > 0
assert len(val_df) > 0
assert len(test_df) > 0

assert train_df["execution_id"].is_unique
assert val_df["execution_id"].is_unique
assert test_df["execution_id"].is_unique

assert set(train_df["execution_id"]).isdisjoint(
    set(val_df["execution_id"])
)

assert set(train_df["execution_id"]).isdisjoint(
    set(test_df["execution_id"])
)

assert set(val_df["execution_id"]).isdisjoint(
    set(test_df["execution_id"])
)

assert (
    train_df["execution_start_time_us"].max()
    < val_df["execution_start_time_us"].min()
)

assert (
    val_df["execution_start_time_us"].max()
    < test_df["execution_start_time_us"].min()
)

assert (
    len(train_df) + len(val_df) + len(test_df)
    == len(execution_features)
)

print("Split integrity checks passed.")


Split integrity checks passed.


In [10]:
split_summary = pd.DataFrame(
    {
        "split": [
            "train",
            "validation",
            "test",
        ],
        "rows": [
            len(train_df),
            len(val_df),
            len(test_df),
        ],
        "start_time_us": [
            train_df["execution_start_time_us"].min(),
            val_df["execution_start_time_us"].min(),
            test_df["execution_start_time_us"].min(),
        ],
        "end_time_us": [
            train_df["execution_start_time_us"].max(),
            val_df["execution_start_time_us"].max(),
            test_df["execution_start_time_us"].max(),
        ],
    }
)

display(split_summary)


,split,rows,start_time_us,end_time_us
0,train,178858,300000000,1812600000000
1,validation,36703,1813200000000,2261100000000
2,test,38024,2262600000000,2678400000000


In [11]:
total_rows = len(execution_features)

actual_split_summary = pd.DataFrame(
    {
        "split": [
            "train",
            "validation",
            "test",
        ],
        "rows": [
            len(train_df),
            len(val_df),
            len(test_df),
        ],
        "fraction": [
            len(train_df) / total_rows,
            len(val_df) / total_rows,
            len(test_df) / total_rows,
        ],
        "percentage": [
            100 * len(train_df) / total_rows,
            100 * len(val_df) / total_rows,
            100 * len(test_df) / total_rows,
        ],
    }
)

display(actual_split_summary)

,split,rows,fraction,percentage
0,train,178858,0.705318,70.531774
1,validation,36703,0.144736,14.473648
2,test,38024,0.149946,14.994578


## Split-Boundary Timestamp Audit

Executions sharing the same start timestamp should not be split across
train/validation or validation/test. This check must pass before the
partitions are frozen or saved.


In [12]:
train_end_time = (
    train_df["execution_start_time_us"].max()
)

val_start_time = (
    val_df["execution_start_time_us"].min()
)

val_end_time = (
    val_df["execution_start_time_us"].max()
)

test_start_time = (
    test_df["execution_start_time_us"].min()
)

train_val_shared_timestamp = (
    train_end_time == val_start_time
)

val_test_shared_timestamp = (
    val_end_time == test_start_time
)

print("Train end:      ", train_end_time)
print("Validation start:", val_start_time)
print("Validation end:  ", val_end_time)
print("Test start:      ", test_start_time)

print(
    "\nSame timestamp across train/validation boundary:",
    train_val_shared_timestamp,
)

print(
    "Same timestamp across validation/test boundary:",
    val_test_shared_timestamp,
)

if (
    train_val_shared_timestamp
    or val_test_shared_timestamp
):
    raise RuntimeError(
        "A split boundary divides executions sharing the same "
        "execution_start_time_us. Do not freeze or save these "
        "partitions yet; update the split logic first."
    )

print(
    "\nSplit-boundary timestamp audit passed."
)


Train end:       1812600000000
Validation start: 1813200000000
Validation end:   2261100000000
Test start:       2262600000000

Same timestamp across train/validation boundary: False
Same timestamp across validation/test boundary: False

Split-boundary timestamp audit passed.


# Next-Occurrence ML Target Construction

The project contract requires a future resource target, but a naïve row-wise shift such as:

```python
groupby("recurrence_key")["peak_cpu_ncu"].shift(-1)
```

is unsafe because multiple reconstructed executions can share the same `recurrence_key` and the same `execution_start_time_us`. Row-wise shifting would arbitrarily treat one simultaneous execution as the next run.

## Target policy used here

1. Define one **recurrence occurrence** as `recurrence_key + execution_start_time_us`.
2. Aggregate all reconstructed executions at that occurrence:
   - CPU occurrence peak = maximum `peak_cpu_ncu`;
   - memory occurrence peak = maximum `peak_memory_normalized`.
3. Order occurrences by their distinct start timestamps.
4. Shift by one **occurrence**, not one execution row.
5. Merge the next-occurrence target back to every execution row in the current occurrence.
6. Construct targets independently inside train, validation, and test so labels never cross split boundaries.

This creates a deterministic, conservative **next distinct recurrence occurrence peak** target. If the team later adopts a different one-to-one execution pairing rule, this target policy must be versioned and all downstream models rerun.


In [13]:
TARGET_CPU_COL = "target_cpu_next_occurrence_peak_ncu"
TARGET_MEMORY_COL = "target_memory_next_occurrence_peak_normalized"
TARGET_TIME_COL = "target_next_occurrence_start_time_us"
TARGET_OCCURRENCE_SIZE_COL = "target_next_occurrence_execution_count"


def add_next_occurrence_targets(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Add next-distinct-occurrence CPU/memory targets."""

    working = df.copy()
    working["_original_row_order"] = np.arange(len(working))

    occurrence_table = (
        working.groupby(
            ["recurrence_key", "execution_start_time_us"],
            as_index=False,
            sort=True,
        )
        .agg(
            occurrence_peak_cpu_ncu=("peak_cpu_ncu", "max"),
            occurrence_peak_memory_normalized=("peak_memory_normalized", "max"),
            occurrence_execution_count=("execution_id", "size"),
        )
        .sort_values(
            ["recurrence_key", "execution_start_time_us"],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    recurrence_groups = occurrence_table.groupby(
        "recurrence_key",
        sort=False,
    )

    occurrence_table[TARGET_TIME_COL] = (
        recurrence_groups["execution_start_time_us"].shift(-1)
    )
    occurrence_table[TARGET_CPU_COL] = (
        recurrence_groups["occurrence_peak_cpu_ncu"].shift(-1)
    )
    occurrence_table[TARGET_MEMORY_COL] = (
        recurrence_groups["occurrence_peak_memory_normalized"].shift(-1)
    )
    occurrence_table[TARGET_OCCURRENCE_SIZE_COL] = (
        recurrence_groups["occurrence_execution_count"].shift(-1)
    )

    target_mapping = occurrence_table[
        [
            "recurrence_key",
            "execution_start_time_us",
            TARGET_TIME_COL,
            TARGET_CPU_COL,
            TARGET_MEMORY_COL,
            TARGET_OCCURRENCE_SIZE_COL,
        ]
    ].copy()

    result = working.merge(
        target_mapping,
        on=["recurrence_key", "execution_start_time_us"],
        how="left",
        validate="many_to_one",
    )

    result = (
        result.sort_values("_original_row_order", kind="mergesort")
        .drop(columns=["_original_row_order"])
        .reset_index(drop=True)
    )

    result["target_cpu_available"] = result[TARGET_CPU_COL].notna()
    result["target_memory_available"] = result[TARGET_MEMORY_COL].notna()

    target_time_available = result[TARGET_TIME_COL].notna()
    if target_time_available.any():
        assert (
            result.loc[target_time_available, TARGET_TIME_COL]
            > result.loc[target_time_available, "execution_start_time_us"]
        ).all()

    return result, occurrence_table


In [14]:
train_df, train_occurrences = add_next_occurrence_targets(train_df)
val_df, val_occurrences = add_next_occurrence_targets(val_df)
test_df, test_occurrences = add_next_occurrence_targets(test_df)

print("Next-occurrence target construction completed.")


Next-occurrence target construction completed.


## Target Audit

This audit reports only target availability/structure. It does not score test performance.


In [15]:
def build_target_audit(
    split_name: str,
    df: pd.DataFrame,
    occurrence_table: pd.DataFrame,
) -> dict[str, object]:
    return {
        "split": split_name,
        "rows": len(df),
        "distinct_occurrences": len(occurrence_table),
        "rows_with_next_occurrence": int(df[TARGET_TIME_COL].notna().sum()),
        "rows_with_cpu_target": int(df[TARGET_CPU_COL].notna().sum()),
        "rows_with_memory_target": int(df[TARGET_MEMORY_COL].notna().sum()),
        "multi_execution_occurrences": int(
            (occurrence_table["occurrence_execution_count"] > 1).sum()
        ),
    }


target_audit = pd.DataFrame(
    [
        build_target_audit("train", train_df, train_occurrences),
        build_target_audit("validation", val_df, val_occurrences),
        build_target_audit("test", test_df, test_occurrences),
    ]
)

display(target_audit)


,split,rows,distinct_occurrences,rows_with_next_occurrence,rows_with_cpu_target,rows_with_memory_target,multi_execution_occurrences
0,train,178858,3473,112480,112480,110255,2277
1,validation,36703,675,16285,16285,16246,513
2,test,38024,731,14834,14834,14834,517


In [16]:
# Target labels must stay inside their frozen split.
assert train_df[TARGET_TIME_COL].dropna().le(
    train_df["execution_start_time_us"].max()
).all()
assert val_df[TARGET_TIME_COL].dropna().le(
    val_df["execution_start_time_us"].max()
).all()
assert test_df[TARGET_TIME_COL].dropna().le(
    test_df["execution_start_time_us"].max()
).all()

print("Target split-isolation checks passed.")


Target split-isolation checks passed.


# Historical Baseline Recommendations

Generate pre-execution recommendations from Notebook 03 point-in-time historical profiles.

Policies: Request, Mean, P90, P95, and P99.

For historical policies, insufficient **resource-specific** history falls back to the current request. `min_runs` is selected using validation only.

Current N03 semantics use `hist_peak_cpu_count` and `hist_peak_memory_count`, which count valid historical execution rows. If the team later changes history depth to distinct recurrence occurrences, this notebook must be rerun.


In [17]:
CPU_BASELINE_COLUMNS = {
    "mean": "hist_peak_cpu_mean",
    "p90": "hist_peak_cpu_p90",
    "p95": "hist_peak_cpu_p95",
    "p99": "hist_peak_cpu_p99",
}

MEMORY_BASELINE_COLUMNS = {
    "mean": "hist_peak_memory_mean",
    "p90": "hist_peak_memory_p90",
    "p95": "hist_peak_memory_p95",
    "p99": "hist_peak_memory_p99",
}

CPU_REQUEST_COL = "requested_cpu_ncu"
MEMORY_REQUEST_COL = "requested_memory_normalized"

CPU_HISTORY_COUNT_COL = "hist_peak_cpu_count"
MEMORY_HISTORY_COUNT_COL = "hist_peak_memory_count"

In [18]:
def add_baseline_recommendations(
    df: pd.DataFrame,
    min_runs: int,
) -> pd.DataFrame:
    """
    Add Request, Mean, P90, P95, and P99 recommendations.

    Historical policies fall back to the current request when
    resource-specific valid history is below min_runs or when the
    historical statistic itself is unavailable.
    """

    if min_runs < 1:
        raise ValueError(
            "min_runs must be at least 1."
        )

    result = df.copy()

    # Request baseline.
    result["rec_cpu_request"] = (
        result[CPU_REQUEST_COL]
    )

    result["rec_memory_request"] = (
        result[MEMORY_REQUEST_COL]
    )

    # Historical CPU policies.
    for policy_name, history_col in (
        CPU_BASELINE_COLUMNS.items()
    ):
        enough_history = (
            result[CPU_HISTORY_COUNT_COL]
            >= min_runs
        )

        historical_value_available = (
            result[history_col].notna()
        )

        use_history = (
            enough_history
            & historical_value_available
        )

        result[f"rec_cpu_{policy_name}"] = (
            result[history_col].where(
                use_history,
                result[CPU_REQUEST_COL],
            )
        )

    # Historical memory policies.
    for policy_name, history_col in (
        MEMORY_BASELINE_COLUMNS.items()
    ):
        enough_history = (
            result[MEMORY_HISTORY_COUNT_COL]
            >= min_runs
        )

        historical_value_available = (
            result[history_col].notna()
        )

        use_history = (
            enough_history
            & historical_value_available
        )

        result[f"rec_memory_{policy_name}"] = (
            result[history_col].where(
                use_history,
                result[MEMORY_REQUEST_COL],
            )
        )

    return result

In [19]:
MIN_RUNS_CANDIDATES = [1, 3, 5, 10]
print("Candidate min_runs values:", MIN_RUNS_CANDIDATES)


Candidate min_runs values: [1, 3, 5, 10]


## Build Validation Candidates


In [20]:
validation_candidates = {}

for min_runs in MIN_RUNS_CANDIDATES:
    validation_candidates[min_runs] = (
        add_baseline_recommendations(
            val_df,
            min_runs=min_runs,
        )
    )

    print(
        f"min_runs={min_runs}:",
        validation_candidates[
            min_runs
        ].shape,
    )

min_runs=1: (36703, 66)
min_runs=3: (36703, 66)
min_runs=5: (36703, 66)
min_runs=10: (36703, 66)


In [21]:
assert all(
    len(candidate_df) == len(val_df)
    for candidate_df
    in validation_candidates.values()
)

print(
    "All min_runs candidates preserve "
    "validation row count."
)

All min_runs candidates preserve validation row count.


In [22]:
fallback_summary_rows = []

for min_runs, candidate_df in (
    validation_candidates.items()
):
    cpu_fallback = (
        candidate_df[
            CPU_HISTORY_COUNT_COL
        ]
        < min_runs
    )

    memory_fallback = (
        candidate_df[
            MEMORY_HISTORY_COUNT_COL
        ]
        < min_runs
    )

    fallback_summary_rows.append(
        {
            "min_runs": min_runs,
            "validation_rows": len(
                candidate_df
            ),
            "cpu_fallback_rows": (
                cpu_fallback.sum()
            ),
            "cpu_fallback_pct": (
                100
                * cpu_fallback.mean()
            ),
            "memory_fallback_rows": (
                memory_fallback.sum()
            ),
            "memory_fallback_pct": (
                100
                * memory_fallback.mean()
            ),
        }
    )

fallback_summary = pd.DataFrame(
    fallback_summary_rows
)

display(fallback_summary)

,min_runs,validation_rows,cpu_fallback_rows,cpu_fallback_pct,memory_fallback_rows,memory_fallback_pct
0,1,36703,7030,19.153748,7489,20.404327
1,3,36703,7966,21.703948,8425,22.954527
2,5,36703,8147,22.197096,8606,23.447675
3,10,36703,9583,26.109582,10042,27.360161


# Shared Evaluator Integration

All policy metrics come from the shared evaluator. The corrected evaluator must expose valid-row denominator counts so missing actual/recommended values are not silently counted as non-violations.


In [23]:
from src.evaluation.evaluator import (
    RecommendationEvaluator,
)

evaluator = RecommendationEvaluator()

BASELINE_POLICIES = {
    "Request": (
        "rec_cpu_request",
        "rec_memory_request",
    ),
    "Mean": (
        "rec_cpu_mean",
        "rec_memory_mean",
    ),
    "P90": (
        "rec_cpu_p90",
        "rec_memory_p90",
    ),
    "P95": (
        "rec_cpu_p95",
        "rec_memory_p95",
    ),
    "P99": (
        "rec_cpu_p99",
        "rec_memory_p99",
    ),
}

ACTUAL_CPU_COL = "peak_cpu_ncu"
ACTUAL_MEMORY_COL = "peak_memory_normalized"

BASELINE_POLICIES

{'Request': ('rec_cpu_request', 'rec_memory_request'),
 'Mean': ('rec_cpu_mean', 'rec_memory_mean'),
 'P90': ('rec_cpu_p90', 'rec_memory_p90'),
 'P95': ('rec_cpu_p95', 'rec_memory_p95'),
 'P99': ('rec_cpu_p99', 'rec_memory_p99')}

In [24]:
sanity_candidate = validation_candidates[1]

sanity_evaluated = evaluator.evaluate(
    data=sanity_candidate,
    actual_cpu_col=ACTUAL_CPU_COL,
    actual_memory_col=ACTUAL_MEMORY_COL,
    recommended_cpu_col="rec_cpu_request",
    recommended_memory_col="rec_memory_request",
    policy_name="Request",
    id_cols=["execution_id", "recurrence_key"],
)

sanity_summary = evaluator.summarize(sanity_evaluated)

EXPECTED_VALID_COUNT_KEYS = {
    "valid_cpu_violation_count",
    "valid_memory_violation_count",
    "valid_any_resource_violation_count",
    "valid_cpu_utilization_count",
    "valid_memory_utilization_count",
}

missing_evaluator_fields = (
    EXPECTED_VALID_COUNT_KEYS - set(sanity_summary.keys())
)

if missing_evaluator_fields:
    raise RuntimeError(
        "The updated shared evaluator is not loaded. Merge the latest main "
        "branch, restart the kernel, and rerun. Missing summary fields: "
        f"{sorted(missing_evaluator_fields)}"
    )

print("Updated evaluator detected.")
display(pd.Series(sanity_summary, name="Request").to_frame())


Updated evaluator detected.


,Request
row_count,36703.000000
valid_cpu_violation_count,36608.000000
valid_memory_violation_count,36141.000000
valid_any_resource_violation_count,36198.000000
valid_cpu_utilization_count,36083.000000
valid_memory_utilization_count,36119.000000
cpu_violation_pct,47.202797
memory_violation_pct,10.613984
any_resource_violation_pct,48.359025
average_cpu_utilization,0.597810


In [25]:
validation_input_missingness = pd.Series(
    {
        "missing_actual_cpu": (
            sanity_candidate[
                ACTUAL_CPU_COL
            ].isna().sum()
        ),
        "missing_actual_memory": (
            sanity_candidate[
                ACTUAL_MEMORY_COL
            ].isna().sum()
        ),
        "missing_request_cpu": (
            sanity_candidate[
                "rec_cpu_request"
            ].isna().sum()
        ),
        "missing_request_memory": (
            sanity_candidate[
                "rec_memory_request"
            ].isna().sum()
        ),
    },
    name="rows",
)

display(
    validation_input_missingness.to_frame()
)

,rows
missing_actual_cpu,0
missing_actual_memory,467
missing_request_cpu,95
missing_request_memory,95


## Evaluate All Validation Configurations


In [26]:
validation_evaluation_rows = []

evaluated_validation = {}

for min_runs in MIN_RUNS_CANDIDATES:

    candidate_df = (
        validation_candidates[min_runs]
    )

    for (
        policy_name,
        (
            cpu_recommendation_col,
            memory_recommendation_col,
        ),
    ) in BASELINE_POLICIES.items():

        evaluated = evaluator.evaluate(
            data=candidate_df,
            actual_cpu_col=ACTUAL_CPU_COL,
            actual_memory_col=ACTUAL_MEMORY_COL,
            recommended_cpu_col=(
                cpu_recommendation_col
            ),
            recommended_memory_col=(
                memory_recommendation_col
            ),
            policy_name=policy_name,
            id_cols=[
                "execution_id",
                "recurrence_key",
            ],
        )

        summary = evaluator.summarize(
            evaluated
        )

        validation_evaluation_rows.append(
            {
                "min_runs": min_runs,
                "policy": policy_name,
                **summary,
            }
        )

        evaluated_validation[
            (
                min_runs,
                policy_name,
            )
        ] = evaluated


validation_comparison = pd.DataFrame(
    validation_evaluation_rows
)

validation_comparison = (
    validation_comparison
    .sort_values(
        [
            "min_runs",
            "policy",
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    validation_comparison
)

,min_runs,policy,row_count,valid_cpu_violation_count,valid_memory_violation_count,valid_any_resource_violation_count,valid_cpu_utilization_count,valid_memory_utilization_count,cpu_violation_pct,memory_violation_pct,any_resource_violation_pct,average_cpu_utilization,average_memory_utilization,average_cpu_waste,average_memory_waste,average_cpu_shortfall,average_memory_shortfall
0,1,Mean,36703,36663,36196,36200,36336,36186,40.525871,34.771798,50.513812,0.689371,0.766777,0.009106,0.001932,0.010024,0.000651
1,1,P90,36703,36663,36196,36200,36336,36186,28.137359,19.253509,36.837017,0.673222,0.725233,0.016179,0.002471,0.007160,0.000345
2,1,P95,36703,36663,36196,36200,36336,36186,26.577203,18.402586,35.425414,0.607205,0.678529,0.018301,0.002676,0.006554,0.000336
3,1,P99,36703,36663,36196,36200,36336,36186,24.176963,11.142115,28.613260,0.467392,0.574661,0.025934,0.002959,0.005957,0.000297
4,1,Request,36703,36608,36141,36198,36083,36119,47.202797,10.613984,48.359025,0.597810,0.441896,0.006351,0.004625,0.015571,0.000917
5,3,Mean,36703,36661,36194,36198,36333,36184,39.938900,32.801017,49.911597,0.686440,0.759069,0.009118,0.001974,0.010369,0.000654
6,3,P90,36703,36661,36194,36198,36333,36184,27.552440,17.281870,36.234046,0.670298,0.717595,0.016191,0.002512,0.007734,0.000355
7,3,P95,36703,36661,36194,36198,36333,36184,25.992199,16.430900,34.822366,0.604277,0.670894,0.018313,0.002718,0.007157,0.000346
8,3,P99,36703,36661,36194,36198,36333,36184,23.591828,9.170028,28.009835,0.464453,0.567025,0.025946,0.003000,0.006583,0.000308
9,3,Request,36703,36608,36141,36198,36083,36119,47.202797,10.613984,48.359025,0.597810,0.441896,0.006351,0.004625,0.015571,0.000917


In [27]:
assert len(
    validation_comparison
) == (
    len(MIN_RUNS_CANDIDATES)
    * len(BASELINE_POLICIES)
)

print(
    "Validation comparison contains",
    len(validation_comparison),
    "configurations."
)

Validation comparison contains 20 configurations.


In [28]:
request_validation_results = (
    validation_comparison[
        validation_comparison["policy"] == "Request"
    ]
    .reset_index(drop=True)
)

# Request must not depend on min_runs.
request_metric_columns = [
    column
    for column in request_validation_results.columns
    if column != "min_runs"
]

request_variation = request_validation_results[
    request_metric_columns
].nunique(dropna=False)

assert (request_variation <= 1).all()

print("Request baseline is invariant across min_runs candidates.")
display(request_validation_results)


Request baseline is invariant across min_runs candidates.


,min_runs,policy,row_count,valid_cpu_violation_count,valid_memory_violation_count,valid_any_resource_violation_count,valid_cpu_utilization_count,valid_memory_utilization_count,cpu_violation_pct,memory_violation_pct,any_resource_violation_pct,average_cpu_utilization,average_memory_utilization,average_cpu_waste,average_memory_waste,average_cpu_shortfall,average_memory_shortfall
0,1,Request,36703,36608,36141,36198,36083,36119,47.202797,10.613984,48.359025,0.59781,0.441896,0.006351,0.004625,0.015571,0.000917
1,3,Request,36703,36608,36141,36198,36083,36119,47.202797,10.613984,48.359025,0.59781,0.441896,0.006351,0.004625,0.015571,0.000917
2,5,Request,36703,36608,36141,36198,36083,36119,47.202797,10.613984,48.359025,0.59781,0.441896,0.006351,0.004625,0.015571,0.000917
3,10,Request,36703,36608,36141,36198,36083,36119,47.202797,10.613984,48.359025,0.59781,0.441896,0.006351,0.004625,0.015571,0.000917


In [29]:
p99_validation_results = (
    validation_comparison[
        validation_comparison["policy"] == "P99"
    ]
    .sort_values("min_runs")
    .reset_index(drop=True)
)

display(
    p99_validation_results[
        [
            "min_runs",
            "valid_cpu_violation_count",
            "valid_memory_violation_count",
            "valid_any_resource_violation_count",
            "cpu_violation_pct",
            "memory_violation_pct",
            "any_resource_violation_pct",
            "average_cpu_utilization",
            "average_memory_utilization",
            "average_cpu_waste",
            "average_memory_waste",
            "average_cpu_shortfall",
            "average_memory_shortfall",
        ]
    ]
)


,min_runs,valid_cpu_violation_count,valid_memory_violation_count,valid_any_resource_violation_count,cpu_violation_pct,memory_violation_pct,any_resource_violation_pct,average_cpu_utilization,average_memory_utilization,average_cpu_waste,average_memory_waste,average_cpu_shortfall,average_memory_shortfall
0,1,36663,36196,36200,24.176963,11.142115,28.613260,0.467392,0.574661,0.025934,0.002959,0.005957,0.000297
1,3,36661,36194,36198,23.591828,9.170028,28.009835,0.464453,0.567025,0.025946,0.003000,0.006583,0.000308
2,5,36661,36194,36198,23.548185,8.868873,27.808166,0.463220,0.564553,0.025901,0.003018,0.006600,0.000308
3,10,36642,36175,36179,23.784182,9.114029,27.764725,0.460755,0.552545,0.025572,0.003350,0.006884,0.000328


# Validation-Based Baseline Selection

TraceAdvisor uses a violation-focused policy objective.

For this Google subset validation run:

- P99 is selected as the historical baseline because it provides the strongest violation protection among Mean/P90/P95/P99.
- `min_runs=5` is selected using validation data only.
- Relative to `min_runs=10`, `min_runs=5` gives lower CPU and memory violation rates, retains more valid evaluated rows, and requires less fallback to Request.
- `min_runs=10` may have a marginally lower any-resource violation percentage, but the difference is very small and comes with higher fallback and slightly worse individual CPU/memory violation rates.

The final test set is **not** used in this decision.


In [30]:
# Define selection BEFORE any metadata or artifact cell uses it.
SELECTED_MIN_RUNS = 5
SELECTED_BASELINE_POLICY = "P99"

selected_validation_result = (
    validation_comparison[
        (validation_comparison["min_runs"] == SELECTED_MIN_RUNS)
        & (validation_comparison["policy"] == SELECTED_BASELINE_POLICY)
    ]
    .reset_index(drop=True)
)

assert len(selected_validation_result) == 1

print("Selected min_runs:", SELECTED_MIN_RUNS)
print("Selected baseline policy:", SELECTED_BASELINE_POLICY)
display(selected_validation_result)


Selected min_runs: 5
Selected baseline policy: P99


,min_runs,policy,row_count,valid_cpu_violation_count,valid_memory_violation_count,valid_any_resource_violation_count,valid_cpu_utilization_count,valid_memory_utilization_count,cpu_violation_pct,memory_violation_pct,any_resource_violation_pct,average_cpu_utilization,average_memory_utilization,average_cpu_waste,average_memory_waste,average_cpu_shortfall,average_memory_shortfall
0,5,P99,36703,36661,36194,36198,36333,36184,23.548185,8.868873,27.808166,0.46322,0.564553,0.025901,0.003018,0.0066,0.000308


## Generate Final Baseline Columns for All Frozen Splits

After the validation-only selection is locked, apply the selected `min_runs` to train, validation, and test artifacts.

This creates recommendation columns in the test artifact, but **does not evaluate or score test performance**.


In [31]:
train_features_final = add_baseline_recommendations(
    train_df,
    min_runs=SELECTED_MIN_RUNS,
)
val_features_final = add_baseline_recommendations(
    val_df,
    min_runs=SELECTED_MIN_RUNS,
)
test_features_final = add_baseline_recommendations(
    test_df,
    min_runs=SELECTED_MIN_RUNS,
)

assert len(train_features_final) == len(train_df)
assert len(val_features_final) == len(val_df)
assert len(test_features_final) == len(test_df)

print("Final baseline columns generated for all frozen splits.")


Final baseline columns generated for all frozen splits.


# Metadata and Canonical Artifact Save


In [32]:
def make_json_serializable(value):
    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, dict):
        return {
            key: make_json_serializable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            make_json_serializable(item)
            for item in value
        ]

    return value

In [33]:
metadata = {
    "dataset": "google_subset_v1",
    "source_artifact": PROFILE_FEATURES_PATH.name,
    "execution_id_field": "execution_id",
    "recurrence_key_field": "recurrence_key",
    "timestamp_field": "execution_start_time_us",
    "requested_cpu_field": "requested_cpu_ncu",
    "requested_memory_field": "requested_memory_normalized",
    "observed_cpu_field": "peak_cpu_ncu",
    "observed_memory_field": "peak_memory_normalized",
    "split": {
        "type": "chronological_timestamp_group_preserving",
        "requested_train_fraction": 0.70,
        "requested_validation_fraction": 0.15,
        "requested_test_fraction": 0.15,
        "actual_train_fraction": len(train_df) / len(execution_features),
        "actual_validation_fraction": len(val_df) / len(execution_features),
        "actual_test_fraction": len(test_df) / len(execution_features),
    },
    "split_boundaries": split_metadata,
    "history_rule": (
        "Historical features use only executions from the same recurrence_key "
        "with execution_end_time_us <= current execution_start_time_us."
    ),
    "min_runs_candidates": MIN_RUNS_CANDIDATES,
    "min_runs_count_basis": (
        "Resource-specific N03 history counts: hist_peak_cpu_count and "
        "hist_peak_memory_count; these currently count valid completed execution rows."
    ),
    "selected_min_runs": SELECTED_MIN_RUNS,
    "selected_baseline_policy": SELECTED_BASELINE_POLICY,
    "baseline_selection_scope": "validation_only",
    "baseline_selection_objective": (
        "Violation-focused. P99 selected as the safest historical baseline. "
        "min_runs=5 selected because it gives lower CPU and memory violation "
        "rates than min_runs=10 while retaining more valid evaluation rows "
        "and requiring less Request fallback."
    ),
    "target_policy_version": "next_distinct_start_occurrence_max_v1",
    "ml_target_definition": (
        "Within each split and recurrence_key, executions sharing "
        "execution_start_time_us form one recurrence occurrence. CPU and memory "
        "peaks are aggregated by max within that occurrence. Targets are the "
        "aggregated peaks of the next distinct occurrence. Targets are shifted "
        "at occurrence level and merged back to execution rows; row-wise shift(-1) "
        "across simultaneous executions is not used."
    ),
    "target_cpu_field": TARGET_CPU_COL,
    "target_memory_field": TARGET_MEMORY_COL,
    "target_timestamp_field": TARGET_TIME_COL,
    "target_split_isolation": (
        "Targets are constructed independently inside train, validation, and test "
        "so labels never cross split boundaries."
    ),
    "test_policy": (
        "Test artifacts may receive deterministic target and recommendation columns, "
        "but N04 does not score test policies or use test results for feature selection, "
        "min_runs selection, threshold selection, model selection, or hyperparameter tuning."
    ),
    "evaluator_requirement": (
        "Shared evaluator exposes valid-row denominator counts and excludes invalid "
        "actual/recommendation pairs from the corresponding metric denominator."
    ),
    "validation_comparison_artifact": VALIDATION_COMPARISON_PATH.name,
    "train_artifact": TRAIN_FEATURES_PATH.name,
    "validation_artifact": VAL_FEATURES_PATH.name,
    "test_artifact": TEST_FEATURES_PATH.name,
}

metadata["selected_validation_metrics"] = (
    selected_validation_result.iloc[0].to_dict()
)

metadata


{'dataset': 'google_subset_v1',
 'source_artifact': 'job_metrics_v0.parquet',
 'execution_id_field': 'execution_id',
 'recurrence_key_field': 'recurrence_key',
 'timestamp_field': 'execution_start_time_us',
 'requested_cpu_field': 'requested_cpu_ncu',
 'requested_memory_field': 'requested_memory_normalized',
 'observed_cpu_field': 'peak_cpu_ncu',
 'observed_memory_field': 'peak_memory_normalized',
 'split': {'type': 'chronological_timestamp_group_preserving',
  'requested_train_fraction': 0.7,
  'requested_validation_fraction': 0.15,
  'requested_test_fraction': 0.15,
  'actual_train_fraction': 0.7053177435573871,
  'actual_validation_fraction': 0.144736478892679,
  'actual_test_fraction': 0.14994577754993393},
 'split_boundaries': {'split_type': 'chronological_timestamp_group_preserving',
  'time_col': 'execution_start_time_us',
  'id_col': 'execution_id',
  'requested_train_fraction': 0.7,
  'requested_val_fraction': 0.15,
  'requested_test_fraction': 0.15000000000000005,
  'total_ro

In [34]:
validation_comparison.to_csv(
    VALIDATION_COMPARISON_PATH,
    index=False,
)

train_features_final.to_parquet(
    TRAIN_FEATURES_PATH,
    index=False,
)
val_features_final.to_parquet(
    VAL_FEATURES_PATH,
    index=False,
)
test_features_final.to_parquet(
    TEST_FEATURES_PATH,
    index=False,
)

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(
        make_json_serializable(metadata),
        file,
        indent=2,
    )

print("Saved:")
print("  Validation comparison:", VALIDATION_COMPARISON_PATH)
print("  Train features:", TRAIN_FEATURES_PATH)
print("  Validation features:", VAL_FEATURES_PATH)
print("  Test features:", TEST_FEATURES_PATH)
print("  Metadata:", METADATA_PATH)


Saved:
  Validation comparison: /Users/sonalilonkar/Desktop/295AB/295A/TraceAdvisor/masters-project/data/processed/google_subset_v1/baseline_validation_comparison.csv
  Train features: /Users/sonalilonkar/Desktop/295AB/295A/TraceAdvisor/masters-project/data/processed/google_subset_v1/train_features.parquet
  Validation features: /Users/sonalilonkar/Desktop/295AB/295A/TraceAdvisor/masters-project/data/processed/google_subset_v1/val_features.parquet
  Test features: /Users/sonalilonkar/Desktop/295AB/295A/TraceAdvisor/masters-project/data/processed/google_subset_v1/test_features.parquet
  Metadata: /Users/sonalilonkar/Desktop/295AB/295A/TraceAdvisor/masters-project/data/processed/google_subset_v1/metadata.json


## Saved Artifact Verification


In [35]:
saved_train = pd.read_parquet(TRAIN_FEATURES_PATH)
saved_val = pd.read_parquet(VAL_FEATURES_PATH)
saved_test = pd.read_parquet(TEST_FEATURES_PATH)

assert len(saved_train) == len(train_features_final)
assert len(saved_val) == len(val_features_final)
assert len(saved_test) == len(test_features_final)

required_saved_columns = [
    TARGET_CPU_COL,
    TARGET_MEMORY_COL,
    TARGET_TIME_COL,
    "rec_cpu_request",
    "rec_cpu_mean",
    "rec_cpu_p90",
    "rec_cpu_p95",
    "rec_cpu_p99",
    "rec_memory_request",
    "rec_memory_mean",
    "rec_memory_p90",
    "rec_memory_p95",
    "rec_memory_p99",
]

for saved_df, split_name in [
    (saved_train, "train"),
    (saved_val, "validation"),
    (saved_test, "test"),
]:
    missing_saved_columns = [
        column
        for column in required_saved_columns
        if column not in saved_df.columns
    ]
    if missing_saved_columns:
        raise RuntimeError(
            f"{split_name} artifact is missing: {missing_saved_columns}"
        )

print("Saved artifact verification passed.")


Saved artifact verification passed.


# N04 Handoff

After this notebook runs successfully:

- the chronological split is frozen;
- split timestamps do not overlap;
- historical profiles remain point-in-time/past-only;
- next-occurrence ML targets are created without row-wise shifting across simultaneous starts;
- baseline recommendations exist for Request/Mean/P90/P95/P99;
- the corrected shared evaluator is used with valid denominators;
- `min_runs=5` and P99 are locked from validation only;
- train/validation/test feature artifacts contain canonical target and baseline recommendation columns;
- test performance has **not** been evaluated.

Downstream N05/N06 should load these prepared artifacts rather than rebuilding historical profiles or targets.
